In [1]:
import os, sys
import pandas as pd
from pathlib import Path

# sys.path.insert(1,os.path.abspath('../..'))
from battle import *

replays_for_test = Path("../data/test_data_replays/")
# replays_1 = Path("../../data/replays/gen9-randombattle")
# replays_2 = Path("../../data/replays/gen9-randombattle_2")
# replays_3 = Path("../../data/replays/gen9-randombattle_3")

def pull_by_num(num, ns="", parse=False):
    return Battle(replays_for_test / f"gen9randombattle-{num}.json", parse)

# # for testing
# bat1 = pull_by_num(2631366401, parse=True)
# bat2 = pull_by_num(2631366781, parse=True)
# bat3 = pull_by_num(2631368887, parse=True)

In [2]:
# these are the column --names--

# info/columns for the entire battle
col_names = [
    'format','id',
    'p1_win', # {0,1}
    'ratedQ','n_turns',
    'start_time','end_time', 'duration',
    'p1name','p1side','p1elo0','p1elo1',
    'p2name','p2side','p2elo0','p2elo1',
    "type_diversity_diff",
    "num_boosting_abilities_diff",
    "num_move_boosters_diff",
    "total_stat_diff",
    "p1_total_adv",
    "p1_revealed_team_size",
    "p2_revealed_team_size"
]

# info/columns for each pokemon,
# format: M<team_i><mon_j>_<thing>
_cols = [
    'name','speciesId','used',
    'gender','shinyQ','level',
    'ability','item','teraType','role',
    'mv1','mv2','mv3','mv4',
    'type1','type2',
    'hp','atk','def',
    'spa','spd','spe',
    'off'
]

for i in range(2):
    for j in range(6):
        col_names.extend([ f"M{i+1}{j+1}_"+s for s in _cols ])

len(col_names)

299

In [3]:
# these are the column --types--

# info/columns for the entire battle
col_types = {
    'format': str,
    'id': int,
    'p1_win': int, # {0,1}
    'ratedQ': bool, 
    'n_turns': int,
    'start_time': int,
    'end_time': int,
    'duration': int,
    'p1name':object, 'p1side':int, 'p1elo0':int, 'p1elo1':int,
    'p2name':str, 'p2side':int, 'p2elo0':int, 'p2elo1':int,
    "type_diversity_diff": int,
    "num_boosting_abilities_diff": int,
    "num_move_boosters_diff": int,
    "total_stat_diff": int,
    "p1_total_adv": int,
    "p1_revealed_team_size": int,
    "p2_revealed_team_size": int
}

# info/columns for each pokemon,
# format: M<team_i><mon_j>_<thing>
_cols = [
    'name','speciesId','used',
    'gender','shinyQ','level',
    'ability','item','teraType','role',
    'mv1','mv2','mv3','mv4',
    'type1','type2',
    'hp','atk','def',
    'spa','spd','spe', 'off'
]
_types = [
    str, str, int,
    str, bool, int,
    str, str, str, str,
    str, str, str, str,
    str, str,
    int, int, int,
    int, int, int, int
]

for i in range(2):
    for j in range(6):
        for k, v in zip(_cols, _types):
            col_types[f"M{i+1}{j+1}_{k}"] = v

len(col_types)

299

In [7]:
with open('../../data/data_col_types.txt','w') as file:
    file.write(str(col_types))

In [8]:
with open('../../data/data_col_names.txt','w') as file:
    file.write(str(col_names))

In [4]:
# ===========================
def Battle_data(bat) : 
    '''
    Returns a long list with data about battle `bat`, players, and individual Pokemon therein.
    The entries correspond to the column names from `data_col_names.txt`
    '''
    # `format`, `id#`
    data = [ 
        bat.formatid, 
        int(bat.id.removeprefix(bat.formatid+'-'))
    ]
    
    # `p1_win` {0,1}
    if bat.winner.name == bat.p1.name : 
        data.append(1)
    else : 
        data.append(0)
    
    # `rated`, `n_turns`, `start_time`, `end_time`, `duration`
    data.extend([bat.rated, len(bat.STATES), bat.start_time, bat.end_time, bat.end_time - bat.start_time])
    
    # `p#name`, `p#side`, `p#elo0`, `p#elo1`
    data.extend(vars(bat.p1).values())
    data.extend(vars(bat.p2).values())

    # `type_diversity_diff`, `num_boosting_abilities_diff`, ..., `p1_total_adv` #21
    data.extend(_aux_battle_data(bat)) # [[2]]

    # `p#_revealed_team_size`
    data.append(len(bat.teams[0].keys()))
    data.append(len(bat.teams[1].keys()))
    
    # getting mon info lists (see below for list of entries)
    # -----------------------
    for i in range(2) :
        team=bat.teams_full[i]
        for M in team.keys() :
            usedQ = int(M in bat.teams[i].keys())
            data.extend(mon_info_list(team[M], usedQ)) # [[1]]
    
    return data


# [[1]]
# ===========================
def mon_info_list(Mon, usedQ):
    '''
    Returns list with entries: 
        'name','speciesId','used',
        'gender','shinyQ','level',
        'ability','item','teraType','role',
        'mv1','mv2','mv3','mv4',
        'type1','type2',
        'hp','atk','def',
        'spa','spd','spe'
    '''
    _L = [Mon['name'], Mon['speciesId'], usedQ]
    
    _keys = [
        'gender','shiny','level','ability',
        'item','teraType','role'
        ]
    _L.extend([ Mon[key] for key in _keys ])

    # movelist
    _moves = Mon['moves']
    _moves.extend([""]*(4-len(_moves))) # pad with "" to get length 4
    _L.extend(_moves)

    # typelist
    _types = Mon['types']
    _types.extend([""]*(2-len(_types))) # pad with "" to get length 2
    _L.extend(_types)
    
    # did this manually b/c it seemed sometimes `off` would be omitted.
    _L.extend([
        Mon['stats']['hp'],
        Mon['stats']['atk'],
        Mon['stats']['def'],
        Mon['stats']['spa'],
        Mon['stats']['spd'],
        Mon['stats']['spe']
    ])
    if Mon['stats'].get('off') != None :
        _L.append(Mon['stats']['off'])
    else:
        _L.append(max(Mon['stats']['atk'], Mon['stats']['spa']))
    
    return _L


# [[2]]
# ===========================
def _aux_battle_data(battle):
    useful_traits = ["num_move_boosters_diff","num_boosting_abilities_diff"]
    stat_names = ['hp','atk','def','spa','spd','spe']
    red_stat_names = ['hp','off','def','spd','spe'] # reduced stat names where off stands in for max(atk,spa)

    _L = []
    
    # Team construction
    team1 = [FullPokemon(battle.teams_full[0][mon]) for mon in battle.teams_full[0].keys()]
    team2 = [FullPokemon(battle.teams_full[1][mon]) for mon in battle.teams_full[1].keys()]
    teams = [team1,team2]
    
    # `type_diversity_diff`
    p1_types = set(mon.types[i] for mon in team1 for i in range(len(mon.types)))
    p2_types = set(mon.types[i] for mon in team2 for i in range(len(mon.types)))
    _L.append(len(p1_types) - len(p2_types))
    
    # `num_boosting_abilities_diff`
    p1_num_boosting_abilities = sum(int(mon.has_boosting_ability()) for mon in team1)
    p2_num_boosting_abilities = sum(int(mon.has_boosting_ability()) for mon in team2)
    _L.append(p1_num_boosting_abilities - p2_num_boosting_abilities)

    # `num_move_boosters_diff`
    p1_num_move_boosters = sum(int(mon.has_boost_move()) for mon in team1)
    p2_num_move_boosters = sum(int(mon.has_boost_move()) for mon in team2)
    _L.append(p1_num_move_boosters - p2_num_move_boosters)
    
    # `total_stat_diff`
    p1_total_stats = sum(sum(mon.stats[stat] for stat in red_stat_names) for mon in team1)
    p2_total_stats = sum(sum(mon.stats[stat] for stat in red_stat_names) for mon in team2)
    _L.append(p1_total_stats - p2_total_stats)
    
    # `p1_total_adv`
    _L.append(sum(FullPokemon.advantage(team1[m1],team2[m2]) for m1 in range(6) for m2 in range(6)))

    return _L

In [5]:
# ===========================
DATA = []
errs = []

for file in replays_for_test.glob("*.json") : 
    try : 
        bat = Battle(file, parse=True)
        DATA.append(Battle_data(bat))
    except : 
        # print(f"error with {file.stem.removeprefix("gen9randombattle-")}")
        errs.append(file.stem.removeprefix("gen9randombattle-"))

len(errs)

0

In [ ]:
with open('../data/data_col_names.txt','r') as file:
    cols = eval(file.read())
cols

In [ ]:
df = pd.DataFrame(DATA, columns=col_names)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4391 entries, 0 to 4390
Columns: 299 entries, format to M26_off
dtypes: bool(13), float64(1), int64(126), object(159)
memory usage: 9.6+ MB


In [9]:
df

,format,id,p1_win,ratedQ,n_turns,start_time,end_time,duration,p1name,p1side,...,M26_mv4,M26_type1,M26_type2,M26_hp,M26_atk,M26_def,M26_spa,M26_spd,M26_spe,M26_off
0,gen9randombattle,2644695683,1,True,27,1783313517,1783313830,313,carboxide_ever,1,...,moongeistbeam,Psychic,Ghost,308,163,166,233,191,177,233
1,gen9randombattle,2645572246,0,True,11,1783453265,1783453339,74,notmetbh102,1,...,wildcharge,Fire,Fighting,322,255,157,216,157,157,255
2,gen9randombattle,2644684138,1,True,20,1783311252,1783311642,390,feraligatr08,1,...,hurricane,Dark,Flying,271,128,177,232,168,211,232
3,gen9randombattle,2645499496,0,True,25,1783444074,1783444636,562,slysh,1,...,foulplay,Normal,Flying,291,222,142,131,142,215,222
4,gen9randombattle,2645496299,0,True,36,1783443699,1783443923,224,xValk_,1,...,shellsmash,Normal,NaN,258,92,120,92,139,196,92
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4386,gen9randombattle,2645293693,1,True,35,1783411250,1783411358,108,botbeater6767,1,...,quiverdance,Bug,Poison,255,114,149,199,174,199,199
4387,gen9randombattle,2645279863,0,True,22,1783408075,1783408276,201,yellowgba,1,...,flareblitz,Fire,Fighting,259,218,164,218,164,224,218
4388,gen9randombattle,2644796835,1,True,11,1783336244,1783336460,216,EraseDataCenters,1,...,discharge,Electric,Water,216,111,223,219,223,188,219
4389,gen9randombattle,2645368940,0,True,29,1783426632,1783426998,366,BeastinD,1,...,bugbuzz,Bug,Electric,263,121,197,288,172,119,288


In [7]:
with open('../data/test_data_cleaned.csv','w') as file:
    file.write(df.to_csv(index=False))

In [8]:
df = pd.read_csv("../data/test_data_cleaned.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4391 entries, 0 to 4390
Columns: 299 entries, format to M26_off
dtypes: bool(13), float64(1), int64(126), object(159)
memory usage: 9.6+ MB


In [ ]:
bat = pull_by_num(2631420781, parse=True)
Battle_data(bat)

In [14]:
ceil(1.5)

2